# Qwen Arbitration for Existing Verification Results

This notebook takes the previous `cluster_existing_proposal_verification_demo` CSV/JSONL as input and runs only Qwen arbitration on samples marked `qwen_needed=True`.

It does not load OWLv2, CLIP/SigLIP, DINO, SAM2, or clustering models.

## 1. Imports and configuration

In [ ]:
from __future__ import annotations

import ast
import gc
import json
import math
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

try:
    import torch
    device = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    torch = None
    device = "cpu"

def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / "notebooks").exists() and (path / "data").exists():
            return path
    return start

REPO_ROOT = find_repo_root()

previous_results_csv = "notebooks/outputs/cluster_existing_proposal_verification_demo/cluster_existing_proposal_verification_results.csv"
previous_results_jsonl = "notebooks/outputs/cluster_existing_proposal_verification_demo/cluster_existing_proposal_verification_results.jsonl"
output_root = "outputs/qwen_arbitrate_existing_verification_results"

QWEN_MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
LOCAL_FILES_ONLY = True
QWEN_MAX_NEW_TOKENS = 384
QWEN_DEVICE_MODE = "cpu"  # "cpu" is slow but avoids 8GB CUDA OOM; use "cuda" only if you have enough VRAM
QWEN_IMAGE_MODE = "debug_only"  # debug_only | crop_only | crop_and_masked | all
MAX_QWEN_SAMPLES = 0  # 0 = all qwen_needed rows

ONLY_QWEN_NEEDED = True
SKIP_ALREADY_ARBITRATED = True
SAVE_PACKETS = True

OUTPUT_ROOT = REPO_ROOT / output_root
PACKET_DIR = OUTPUT_ROOT / "qwen_packets"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
PACKET_DIR.mkdir(parents=True, exist_ok=True)

print("repo root:", REPO_ROOT)
print("device:", device)
print("output:", OUTPUT_ROOT)

## 2. Load previous verification results

In [ ]:
def repo_path(path: str | Path) -> Path:
    p = Path(path)
    if p.is_absolute():
        return p
    if p.exists():
        return p
    return REPO_ROOT / p

csv_path = repo_path(previous_results_csv)
jsonl_path = repo_path(previous_results_jsonl)

if csv_path.exists():
    df = pd.read_csv(csv_path)
elif jsonl_path.exists():
    rows = [json.loads(line) for line in jsonl_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    df = pd.DataFrame(rows)
else:
    raise FileNotFoundError(f"Could not find previous results CSV/JSONL: {csv_path} or {jsonl_path}")

print("loaded:", df.shape)
print("columns:", list(df.columns))

work_df = df.copy()
if ONLY_QWEN_NEEDED and "qwen_needed" in work_df.columns:
    work_df = work_df[work_df["qwen_needed"].astype(str).str.lower().isin({"true", "1"})].copy()
if SKIP_ALREADY_ARBITRATED and "qwen_ran" in work_df.columns:
    work_df = work_df[~work_df["qwen_ran"].astype(str).str.lower().isin({"true", "1"})].copy()
if MAX_QWEN_SAMPLES and len(work_df) > MAX_QWEN_SAMPLES:
    work_df = work_df.head(MAX_QWEN_SAMPLES).copy()

print("qwen arbitration rows:", len(work_df))
display(work_df[["proposal_id", "dino_label", "dino_score", "conflict_type", "quality_status", "debug_vis_path"]].head(20))

## 3. Image loading helpers

In [ ]:
def parse_maybe_list(value):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return None
    if isinstance(value, (list, tuple, np.ndarray)):
        return [float(x) for x in value[:4]]
    text = str(value).strip()
    if not text:
        return None
    for parser in (json.loads, ast.literal_eval):
        try:
            out = parser(text)
            if isinstance(out, (list, tuple)) and len(out) >= 4:
                return [float(x) for x in out[:4]]
        except Exception:
            pass
    return None

def is_missing(value):
    if value is None:
        return True
    try:
        return bool(pd.isna(value))
    except Exception:
        return False

def resolve_existing(path_value):
    if is_missing(path_value):
        return None
    text = str(path_value).strip()
    if not text or text.lower() in {"nan", "none"}:
        return None
    p = Path(text)
    candidates = [p]
    if not p.is_absolute():
        candidates.extend([REPO_ROOT / p, REPO_ROOT / "notebooks" / p])
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None

def load_rgb(path_value):
    path = resolve_existing(path_value)
    if not path:
        return None
    try:
        return Image.open(path).convert("RGB")
    except Exception:
        return None

def load_mask(path_value):
    path = resolve_existing(path_value)
    if not path:
        return None
    try:
        return np.array(Image.open(path).convert("L")) > 0
    except Exception:
        return None

def crop_from_bbox(frame, bbox):
    if frame is None or bbox is None:
        return None
    w, h = frame.size
    x1, y1, x2, y2 = [int(round(float(v))) for v in bbox]
    x1, y1, x2, y2 = max(0, x1), max(0, y1), min(w, x2), min(h, y2)
    if x2 <= x1 or y2 <= y1:
        return None
    return frame.crop((x1, y1, x2, y2))

def make_masked_crop(frame, bbox, mask):
    crop = crop_from_bbox(frame, bbox)
    if crop is None or mask is None or bbox is None:
        return crop
    x1, y1, x2, y2 = [int(round(float(v))) for v in bbox]
    x1, y1, x2, y2 = max(0, x1), max(0, y1), min(frame.size[0], x2), min(frame.size[1], y2)
    m = mask[y1:y2, x1:x2]
    arr = np.array(crop).copy()
    if m.shape[:2] == arr.shape[:2]:
        arr[~m] = 96
    return Image.fromarray(arr)

def load_images_for_row(row):
    frame = load_rgb(row.get("frame_path"))
    crop = load_rgb(row.get("crop_path"))
    bbox = parse_maybe_list(row.get("bbox_xyxy"))
    if crop is None:
        crop = crop_from_bbox(frame, bbox)
    mask = load_mask(row.get("mask_path"))
    masked = make_masked_crop(frame, bbox, mask) if frame is not None and bbox is not None and mask is not None else crop
    debug = load_rgb(row.get("debug_vis_path"))
    return frame, crop, masked, debug

## 4. Load Qwen only

In [ ]:
qwen_model = None
qwen_processor = None
qwen_load_error = ""

try:
    if torch is not None and torch.cuda.is_available():
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
    qwen_device = "cuda" if QWEN_DEVICE_MODE == "cuda" and torch is not None and torch.cuda.is_available() else "cpu"
    dtype = torch.float16 if qwen_device == "cuda" else torch.float32
    print(f"Loading Qwen only: {QWEN_MODEL_ID} on {qwen_device} (QWEN_DEVICE_MODE={QWEN_DEVICE_MODE})")
    qwen_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        QWEN_MODEL_ID,
        torch_dtype=dtype,
        device_map="auto" if qwen_device == "cuda" else None,
        local_files_only=LOCAL_FILES_ONLY,
    ).eval()
    if qwen_device == "cpu":
        qwen_model = qwen_model.to(qwen_device)
    qwen_processor = AutoProcessor.from_pretrained(QWEN_MODEL_ID, local_files_only=LOCAL_FILES_ONLY)
    print("Qwen loaded.")
except Exception as exc:
    qwen_load_error = f"{type(exc).__name__}: {exc}"
    print("Qwen unavailable:", qwen_load_error)

## 5. Qwen arbitration function

In [ ]:
def qwen_prompt(row: dict) -> str:
    evidence = {
        "proposal_id": row.get("proposal_id"),
        "dino_label": row.get("dino_label"),
        "dino_score": row.get("dino_score"),
        "cluster_suggested_label": row.get("cluster_suggested_label"),
        "objectness_status": row.get("objectness_status"),
        "objectness_score": row.get("objectness_score"),
        "mask_quality_score": row.get("mask_quality_score"),
        "owlv2_crop_top1_label": row.get("owlv2_crop_top1_label"),
        "owlv2_crop_top1_score": row.get("owlv2_crop_top1_score"),
        "owlv2_crop_top5_labels": row.get("owlv2_crop_top5_labels"),
        "verifier_top1_label": row.get("verifier_top1_label"),
        "verifier_top1_score": row.get("verifier_top1_score"),
        "verifier_top5_labels": row.get("verifier_top5_labels"),
        "verifier_margin": row.get("verifier_margin"),
        "pre_qwen_final_label": row.get("final_fine_label"),
        "pre_qwen_quality": row.get("quality_status"),
        "conflict_type": row.get("conflict_type"),
        "decision_reasons": row.get("decision_reasons"),
    }
    return f"""
You are arbitrating one existing DINO+SAM2 object proposal from an egocentric video frame.

Important tasks:
1. Decide whether the proposal is a real object, uncertain object, or fake/background.
2. Decide whether the disagreement is caused by mask/crop quality: under-segmented, over-segmented, merged instances, background false positive, crop too unclear, or mask missing.
3. If mask/crop quality is too bad, do not force a label. Raise mask_alert.
4. If it is a real object and visible enough, arbitrate using DINO/SAM2, OWLv2, CLIP/SigLIP, cluster evidence, and your own visual/context judgment.
5. Use the full frame context when helpful. For example, an object at the sink edge may plausibly be a faucet; a thin dark line on a cutting board may be board edge/shadow rather than a knife.

Evidence:
{json.dumps(evidence, ensure_ascii=False, indent=2, default=str)}

Return JSON only:
{{
  "is_object": true,
  "objectness_decision": "real_object | uncertain_object | fake_object",
  "mask_issue": "none | under_segmented | over_segmented | merged_instances | background_false_positive | crop_too_unclear | mask_missing",
  "mask_issue_severity": "none | low | medium | high",
  "needs_mask_alert": false,
  "context_reasoning": "short explanation using image context/location",
  "model_conflict_summary": "short explanation of disagreement",
  "arbitrated_label": null,
  "arbitrated_coarse_label": null,
  "decision": "accept_label | relabel | uncertain | reject | mask_alert",
  "confidence": 0.0,
  "reason": "short final reason"
}}
""".strip()

def parse_json_output(text: str) -> dict:
    cleaned = text.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.strip("`").removeprefix("json").strip()
    start, end = cleaned.find("{"), cleaned.rfind("}")
    if start >= 0 and end > start:
        cleaned = cleaned[start:end + 1]
    return json.loads(cleaned)

def select_qwen_images(frame, crop, masked, debug):
    mode = QWEN_IMAGE_MODE
    if mode == "debug_only":
        return [im for im in [debug, crop] if im is not None][:1]
    if mode == "crop_only":
        return [im for im in [crop, debug] if im is not None][:1]
    if mode == "crop_and_masked":
        return [im for im in [crop, masked, debug] if im is not None][:2]
    return [im for im in [frame, crop, masked, debug] if im is not None]

def run_qwen_row(row: dict) -> dict:
    if qwen_model is None or qwen_processor is None:
        return {"qwen_error": qwen_load_error or "Qwen not loaded"}
    frame, crop, masked, debug = load_images_for_row(row)
    images = select_qwen_images(frame, crop, masked, debug)
    content = [{"type": "image", "image": im} for im in images]
    content.append({"type": "text", "text": qwen_prompt(row)})
    messages = [{"role": "user", "content": content}]
    try:
        text = qwen_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = qwen_processor(text=[text], images=images if images else None, return_tensors="pt").to(qwen_device if "qwen_device" in globals() else device)
        with torch.inference_mode():
            generated = qwen_model.generate(**inputs, max_new_tokens=QWEN_MAX_NEW_TOKENS, do_sample=False)
        trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated)]
        output = qwen_processor.batch_decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
        parsed = parse_json_output(output)
        parsed["raw_qwen_output"] = output
        return parsed
    except Exception as exc:
        return {"qwen_error": f"{type(exc).__name__}: {exc}"}

## 6. Run Qwen arbitration

In [ ]:
def apply_qwen(row: dict, q: dict) -> dict:
    out = dict(row)
    out["qwen_ran"] = bool(q and not q.get("qwen_error"))
    out["qwen_error"] = q.get("qwen_error") if q else None
    out["qwen_objectness_decision"] = q.get("objectness_decision") if q else None
    out["qwen_mask_issue"] = q.get("mask_issue") if q else None
    out["qwen_needs_mask_alert"] = bool(q.get("needs_mask_alert")) if q else False
    out["qwen_context_reasoning"] = q.get("context_reasoning") if q else None
    out["qwen_model_conflict_summary"] = q.get("model_conflict_summary") if q else None
    out["qwen_arbitrated_label"] = q.get("arbitrated_label") if q else None
    out["qwen_decision"] = q.get("decision") if q else None
    out["qwen_confidence"] = q.get("confidence") if q else None
    out["qwen_reason"] = q.get("reason") if q else None
    out["final_after_qwen_label"] = row.get("final_fine_label")
    out["quality_after_qwen"] = row.get("quality_status")
    decision = str(q.get("decision") or "") if q else ""
    conf = float(q.get("confidence") or 0.0) if q else 0.0
    if decision == "mask_alert" or out["qwen_needs_mask_alert"]:
        out["quality_after_qwen"] = "uncertain"
    elif decision == "reject" or q.get("objectness_decision") == "fake_object":
        out["final_after_qwen_label"] = "background_candidate"
        out["quality_after_qwen"] = "rejected"
    elif decision in {"relabel", "accept_label"} and q.get("arbitrated_label") and conf >= 0.65:
        out["final_after_qwen_label"] = str(q["arbitrated_label"]).strip().lower()
        out["quality_after_qwen"] = "high_quality" if conf >= 0.75 else "uncertain"
    elif decision == "uncertain":
        out["quality_after_qwen"] = "uncertain"
    return out

rows = []
jsonl_path = OUTPUT_ROOT / "qwen_arbitration_results.jsonl"
with jsonl_path.open("w", encoding="utf-8") as fh:
    for _, row in tqdm(work_df.iterrows(), total=len(work_df)):
        rowd = row.to_dict()
        packet = PACKET_DIR / f"proposal_{int(rowd['proposal_id']):07d}"
        packet.mkdir(parents=True, exist_ok=True)
        q = run_qwen_row(rowd)
        merged = apply_qwen(rowd, q)
        rows.append(merged)
        (packet / "qwen_response.json").write_text(json.dumps(q, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
        fh.write(json.dumps(merged, ensure_ascii=False, default=str) + "\n")

qwen_df = pd.DataFrame(rows)
csv_out = OUTPUT_ROOT / "qwen_arbitration_results.csv"
qwen_df.to_csv(csv_out, index=False, encoding="utf-8")
print("wrote", csv_out)
display(qwen_df[["proposal_id", "dino_label", "conflict_type", "qwen_ran", "qwen_error", "qwen_decision", "qwen_mask_issue", "qwen_arbitrated_label", "qwen_confidence", "final_after_qwen_label", "quality_after_qwen", "qwen_reason"]].head(30))

## 7. Summary

In [ ]:
summary = {
    "input_rows": int(len(df)),
    "qwen_rows": int(len(qwen_df)),
    "qwen_decision_counts": qwen_df["qwen_decision"].value_counts(dropna=False).to_dict() if "qwen_decision" in qwen_df else {},
    "qwen_mask_issue_counts": qwen_df["qwen_mask_issue"].value_counts(dropna=False).to_dict() if "qwen_mask_issue" in qwen_df else {},
    "quality_after_qwen_counts": qwen_df["quality_after_qwen"].value_counts(dropna=False).to_dict() if "quality_after_qwen" in qwen_df else {},
    "rescued_uncertain_to_high_quality": int(((qwen_df["quality_status"] == "uncertain") & (qwen_df["quality_after_qwen"] == "high_quality")).sum()) if len(qwen_df) else 0,
    "rejected_by_qwen": int((qwen_df["quality_after_qwen"] == "rejected").sum()) if len(qwen_df) else 0,
    "mask_alerts": int(qwen_df["qwen_needs_mask_alert"].fillna(False).astype(bool).sum()) if len(qwen_df) else 0,
}
(OUTPUT_ROOT / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
pd.DataFrame([summary]).to_csv(OUTPUT_ROOT / "summary.csv", index=False)
display(pd.Series(summary["qwen_decision_counts"], name="qwen_decision").to_frame())
display(pd.Series(summary["qwen_mask_issue_counts"], name="qwen_mask_issue").to_frame())
display(pd.Series(summary["quality_after_qwen_counts"], name="quality_after_qwen").to_frame())
print("rescued_uncertain_to_high_quality:", summary["rescued_uncertain_to_high_quality"])
print("rejected_by_qwen:", summary["rejected_by_qwen"])
print("mask_alerts:", summary["mask_alerts"])